# Инструменты search, open, read на живой Википедии

Смотрим, что модель получит в ответ на вызовы: выдачу поиска, введение с оглавлением и текст раздела.
Ответы Википедии сохраняются в `data/sources/wiki_cache/`: повторный запуск берёт их оттуда, без сети.
Нужен контакт для User-Agent (переменная `WIKI_CONTACT` или строка ниже).

In [2]:
import os, sys, time
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import WikiTools

CONTACT = os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")  # или вписать сюда почту / ссылку на проект
client = WikiClient(contact=CONTACT)
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache", client)
tools = WikiTools(cache)

def call(name, argument):
    """Вызов как у модели: печатаем ответ, его длину и сколько обращений к Википедии потребовалось."""
    before, failed, start = client.requests, len(client.failures), time.time()
    result = tools.call(name, argument)
    print(f"--- {name}({argument!r}): {len(result)} символов, запросов к Википедии (с повторами): "
          f"{client.requests - before}, {time.time() - start:.1f} с")
    for reason in client.failures[failed:]:
        print("    неудачная попытка:", reason)
    print()
    print(result)
    return result

In [3]:
found = call("search", "градиентный спуск")

--- search('градиентный спуск'): 1456 символов, запросов к Википедии: 4, 74.2 с

Результаты поиска «градиентный спуск»:
d1 | ru | Градиентный спуск | Градиентный спуск, метод градиентного спуска — численный метод нахождения локального минимума или максимума функции с помощью движения вдоль градиента, один из основных численных методов современной…
d2 | ru | Стохастический градиентный спуск | Стохастический градиентный спуск (англ. Stochastic gradient descent, SGD) — итерационный метод для оптимизации целевой функции с подходящими свойствами гладкости (например, дифференцируемость или…
d3 | ru | Метод обратного распространения ошибки | Метод обратного распространения ошибки — метод вычисления градиента, который используется при обновлении весов многослойного перцептрона. Впервые метод был описан в 1974 году Александром Галушкиным,…
d4 | en | Gradient descent | Gradient descent is a method for unconstrained mathematical optimization. It is a first-order iterative algorithm for minimizing

In [4]:
toc = call("open", "d1")

--- open('d1'): 2238 символов, запросов к Википедии: 0, 0.0 с

d1 | ru | Градиентный спуск
d1#0 Введение
Градиентный спуск, метод градиентного спуска — численный метод нахождения локального минимума или максимума функции с помощью движения вдоль градиента, один из основных численных методов современной оптимизации.
Активно используется в вычислительной математике не только для непосредственного решения задач оптимизации (минимизации), но и для задач, которые могут быть переписаны на языке оптимизации (решение нелинейных уравнений, поиск равновесий, обратные задачи и т. д.). Метод градиентного спуска можно использовать для задач оптимизации в бесконечномерных пространствах, например, для численного решения задач оптимального управления.
Особенно большой интерес к градиентным методам в последние годы связан с тем, что градиентные спуски и их стохастические / рандомизированные варианты лежат в основе почти всех современных алгоритмов обучения, разрабатываемых в анализе данных.
d1#1 Описан

In [5]:
# Первый раздел после введения из оглавления.
section = next(line.split()[0] for line in toc.splitlines()[1:] if line.startswith("d1#") and not line.startswith("d1#0"))
text = call("read", section)

--- read('d1#1'): 1239 символов, запросов к Википедии: 0, 0.0 с

d1#1 | ru | Градиентный спуск — Описание
Пусть целевая функция имеет вид: $F({\vec {x}}):\;\mathbb {X} \to \mathbb {R}$.
И задача оптимизации задана следующим образом: $F({\vec {x}})\to \min _{{\vec {x}}\in \mathbb {X} }$
В случае, когда требуется найти максимум, вместо $F({\vec {x}})$ используется $-F({\vec {x}})$
Основная идея метода заключается в том, чтобы идти в направлении наискорейшего спуска, а это направление задаётся антиградиентом $-\nabla F$: ${\vec {x}}^{[j+1]}={\vec {x}}^{[j]}-\lambda ^{[j]}\nabla F\left({\vec {x}}^{[j]}\right)$
где $\lambda ^{[j]}$
задает скорость градиентного спуска и может быть выбрана

постоянной (в этом случае метод может расходиться);
убывающей в процессе градиентного спуска;
гарантирующей наискорейший спуск:
Для поиска минимума $F\left({\vec {x}}\right)$ получаем $\lambda ^{[j]}=\mathrm {argmin} _{\lambda }F\left({\vec {x}}^{[j+1]}\right)=\mathrm {argmin} _{\lambda }\,F\left({\vec {x}

In [6]:
# Запрос латиницей — поиск по английской Википедии; статьи, найденные раньше, сохраняют свои id.
found_en = call("search", "gradient descent")

--- search('gradient descent'): 1407 символов, запросов к Википедии: 2, 1.6 с

Результаты поиска «gradient descent»:
d4 | en | Gradient descent | Gradient descent is a method for unconstrained mathematical optimization. It is a first-order iterative algorithm for minimizing a differentiable multivariate function. The idea is to take repeated…
d5 | en | Stochastic gradient descent | Stochastic gradient descent (often abbreviated SGD) is an iterative method for optimizing an objective function with suitable smoothness properties (e.g. differentiable or subdifferentiable). It can…
d7 | en | Conjugate gradient method | In mathematics, the conjugate gradient method is an algorithm for the numerical solution of particular systems of linear equations, namely those whose matrix is positive-semidefinite. The conjugate…
d1 | ru | Градиентный спуск | Градиентный спуск, метод градиентного спуска — численный метод нахождения локального минимума или максимума функции с помощью движения вдоль градиен

In [7]:
# Источники: что можно указать в #SOURCE и что подставит сборщик.
for ref in ["d1", section, "d2#1", "d9"]:
    print(ref, tools.valid_source(ref), tools.source(ref))
print()
print(tools.table())
print("вызовов:", len(tools.calls))

d1 True {'lang': 'ru', 'title': 'Градиентный спуск', 'url': 'https://ru.wikipedia.org/wiki/%D0%93%D1%80%D0%B0%D0%B4%D0%B8%D0%B5%D0%BD%D1%82%D0%BD%D1%8B%D0%B9_%D1%81%D0%BF%D1%83%D1%81%D0%BA', 'revid': 142334438, 'section': ''}
d1#1 True {'lang': 'ru', 'title': 'Градиентный спуск', 'url': 'https://ru.wikipedia.org/wiki/%D0%93%D1%80%D0%B0%D0%B4%D0%B8%D0%B5%D0%BD%D1%82%D0%BD%D1%8B%D0%B9_%D1%81%D0%BF%D1%83%D1%81%D0%BA#%D0%9E%D0%BF%D0%B8%D1%81%D0%B0%D0%BD%D0%B8%D0%B5', 'revid': 142334438, 'section': 'Описание'}
d2#1 False None
d9 False None

{'d1': {'lang': 'ru', 'title': 'Градиентный спуск', 'pageid': 388413, 'opened': True, 'read': [1]}, 'd2': {'lang': 'ru', 'title': 'Стохастический градиентный спуск', 'pageid': 8223477, 'opened': False, 'read': []}, 'd3': {'lang': 'ru', 'title': 'Метод обратного распространения ошибки', 'pageid': 361924, 'opened': False, 'read': []}, 'd4': {'lang': 'en', 'title': 'Gradient descent', 'pageid': 201489, 'opened': False, 'read': []}, 'd5': {'lang': 'en', 'tit

In [8]:
# Повтор того же эпизода: всё берётся из сохранённого, запросов к Википедии — 0.
tools.reset()
_ = call("search", "градиентный спуск")
_ = call("open", "d1")

--- search('градиентный спуск'): 1456 символов, запросов к Википедии: 0, 0.0 с

Результаты поиска «градиентный спуск»:
d1 | ru | Градиентный спуск | Градиентный спуск, метод градиентного спуска — численный метод нахождения локального минимума или максимума функции с помощью движения вдоль градиента, один из основных численных методов современной…
d2 | ru | Стохастический градиентный спуск | Стохастический градиентный спуск (англ. Stochastic gradient descent, SGD) — итерационный метод для оптимизации целевой функции с подходящими свойствами гладкости (например, дифференцируемость или…
d3 | ru | Метод обратного распространения ошибки | Метод обратного распространения ошибки — метод вычисления градиента, который используется при обновлении весов многослойного перцептрона. Впервые метод был описан в 1974 году Александром Галушкиным,…
d4 | en | Gradient descent | Gradient descent is a method for unconstrained mathematical optimization. It is a first-order iterative algorithm for minimizing 

In [9]:
# Новый запрос — проверить, повторяется ли задержка (по старым запросам ответы уже сохранены).
_ = call("search", "фотосинтез")


--- search('фотосинтез'): 1391 символов, запросов к Википедии: 2, 1.6 с

Результаты поиска «фотосинтез»:
d7 | ru | Фотосинтез | Фотоси́нтез (от др.-греч. φῶς — «свет» и σύνθεσις — «соединённый», «складной», «связывание», «синтез») — сложный химический процесс преобразования энергии видимого света (в некоторых случаях —…
d8 | ru | C4-фотосинтез | C4-фотосинтез, или цикл Хэтча — Слэка, — путь связывания углерода, характерный для высших растений, первым продуктом которого является четырёхуглеродная щавелевоуксусная кислота, а не трёхуглеродная…
d9 | ru | CAM-фотосинтез | CAM-фотоси́нтез (англ. crassulacean acid metabolism (CAM), «кисло́тный метаболи́зм толстя́нковых» или «метаболи́зм углеро́да по ти́пу толстя́нковых») — метаболический путь связывания углерода,…
d10 | en | Photosynthesis | Photosynthesis is a system of biological processes by which photopigment-bearing autotrophic organisms, such as most plants, algae and cyanobacteria, convert light energy—typically from sunlight—into…
d1